# SALM Fine-tuning 13 · Accents in the wild: conversational English (EdAcc)

> **Task** speech → verbatim transcript · **Data** EdAcc (Edinburgh International Accents of English Corpus)
> **Metric** WER ↓ · **Result** 18.37% → **14.96%** (−18.6%) · **Time** ~2 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: friends chatting, in 40 kinds of English

EdAcc records video calls between friends. Speakers come from Nigeria, Kenya, India, Jamaica, Vietnam, Chile, Scotland
and beyond: 40 accents and 26 first languages. It has everything a read-speech model finds hard: overlapping turns,
backchannels ("mm hmm", "yeah"), fillers, laughter, and segments from 0.2 seconds to 9 minutes long.

The base model is already decent here (16-18% WER, depending on the prompt). This tutorial is mostly about **measuring
correctly**. The benchmark has three pitfalls, and each one distorts the result if you don't check for it:

| pitfall | what you'd see | what's really going on | the check |
|---|---|---|---|
| 1. greedy repetition loops | a fine-tune (or baseline) much worse than expected | 2-3 clips looping `the the the …` to the token cap | cap tokens by audio duration |
| 2. `IGNORE_TIME_SEGMENT_IN_SCORING` | a large fine-tuning gain | the model learns to *say* the scoring marker | drop marker rows from every split |
| 3. `mm` → "Ouais." | an error floor on short clips | the prompt doesn't say the audio is English | name the language in the prompt |

Fine-tuning gains a moderate amount here, and part of the headline number comes from the prompt: against the
test-oracle prompt, the gain is −7.7%. The notebook also shows why most of the remaining errors are annotation
conventions rather than recognition errors.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data, and pitfall 2: a reference that says "ignore me"

EdAcc has no training split. The recipe trains on its *validation* set (speaker-disjoint from test), holds 15% of the
conversations out as dev, and reports on the official test set.

Every conversation begins with both speakers reading a fixed passage ("Please call Stella…"). The corpus doesn't
transcribe it: its reference is the literal string `IGNORE_TIME_SEGMENT_IN_SCORING`, the NIST convention for
"don't score this". If these rows are kept:

* the **base model** transcribes the passage: thousands of words on test, all counted as insertions against a
  one-word reference;
* a **fine-tune** sees the marker 25 times in training and learns to output it verbatim on nearly all of the 61 test
  segments.

That gap is worth several WER points, enough to make a marginal real improvement look like a double-digit one.
`prepare_benchmarks.py edacc` drops these segments from every split, as NIST and ESB scoring do.

> Before measuring any ASR benchmark, grep its references for scoring markers. When a gain concentrates in a few rows,
> read those rows.

In [ ]:
d = M / "edacc"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py edacc", log="prep_edacc.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
for s in ("train", "dev", "test"):
    rows = [json.loads(l) for l in open(d / f"{s}.json")]
    short = sum(r["duration"] < 3 for r in rows) / len(rows)
    print(f"{s:5s} {len(rows):5d} segments, {sum(r['duration'] for r in rows) / 3600:5.2f} h, {short:.0%} shorter than 3 s, "
          f"{len({r['accent'] for r in rows})} accents; markers left: {sum('IGNORE_TIME' in r['text'] for r in rows)}")

## 2. Baseline, and pitfall 1: greedy loops

Test segments run up to 536 s, so decoding needs `--max-tokens 2048`. On a few clips, greedy decoding can fall into a
loop (`I I I I …`, `super super super …`) and run all the way to 2,048 tokens: thousands of inserted words from 2-3
clips. This affects the base model *and* the fine-tunes. `--max-tokens-per-sec 10` caps each request at 10 tokens
per second of audio (+32), far above any speaking rate, so a loop costs a few words instead of thousands. Every decode
in this notebook uses the cap, base and fine-tuned alike.

Pitfall 3 is the prompt. The first two prompts don't say which language to expect, and on short clips the model
drifts: `mm` becomes "Ouais.", `yeah` becomes "sí", `uh yes` comes out in Korean. Naming English (and the
backchannels) helps on test, but on dev the three prompts land within 0.2 points of each other and `p1` wins, so `p1`
is the prompt chosen on dev. Expected: `p1` 13.51% dev / **18.37%** test (the baseline), `p2` 13.60% / 16.58%, `p4`
13.69% / 16.20% (the test-oracle).

In [ ]:
TASK = "--task asr --max-tokens 2048 --max-tokens-per-sec 10 --max-model-len 16384"
PROMPTS = {
    "p1": "Provide a verbatim transcript of the audio.",
    "p2": "Transcribe the audio.",
    "p4": "Transcribe this English conversation verbatim, including fillers and backchannels such as um, uh, mm hmm "
          "and yeah.",
}
base = {}
for p, text in PROMPTS.items():
    for s in ("dev", "test"):
        base[p, s] = 100 * evaluate(CKPT, d / f"{s}.json", f"acc_base_{p}_{s}", TASK, f"--prompt {q(text)}")["score"]
    print(f"{p}  dev WER {base[p, 'dev']:5.2f}%   test WER {base[p, 'test']:5.2f}%   {text}")
BEST_P = min(PROMPTS, key=lambda p: base[p, "dev"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]  # its test score: the baseline fine-tuning is compared with
ORACLE = min(base[p, "test"] for p in PROMPTS)  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}% WER")

Where do the errors live? Split by segment length:

In [ ]:
def by_length(name):
    """WER per segment-length bucket (task_metrics.py --by-duration)."""
    cmd = [VLLM_PYTHON, str(FT / "task_metrics.py"), "--task", "asr", "--hyps", str(EVAL / f"{name}.jsonl"), "--by-duration"]
    print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)


by_length("acc_base_p4_test")
samples("acc_base_p1_test", n=8)

Half the test segments are shorter than 3 s, and there the WER is ~49% (shown for `p4`, the training prompt). They're
mostly backchannels (`mm hmm`, `yeah`, `hmm`), often cut to 0.2-0.5 s. The model deletes them, renders them in another
language, or picks a different backchannel than the annotator did. Long segments are already at ~10-11%. `p1`, the
prompt chosen on dev, scores about the same on short clips but worse on segments over 30 s (16.7% vs 10.0%), which is
most of its gap to `p4` on test.

## 3. Recipe: in-domain conventions, gently

The training targets are lower-cased (EdAcc's references are all upper case; the scorer ignores case anyway), and the
prompt is the English-naming `p4`. Most of what the fine-tune can learn is EdAcc's *transcription conventions*:
which backchannel spelling, `gonna` vs `going to`, keeping fillers. That happens within ~100 steps. After that the
model starts overfitting 9.8 hours of audio, so the recipe uses LR 5e-5 and evaluates early checkpoints.

In [ ]:
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(PROMPTS['p4'])} {pairs}")
CFG = WORK / "configs" / "acc.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'acc'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name acc_dev "
   f"--train-encoder --train-proj --lr 5e-5 --warmup-steps 40 --max-steps 400 "
   f"--batch-tokens 24000 --limit-train-batches 20 --val-every-n-epochs 2 --limit-val-batches 50 --save-top-k -1")

## 4. Train (~15 min)

`--max-steps 400` sets the learning-rate schedule; training stops at step 200 because the best checkpoints are early.

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 40 120 trainer.max_steps=200")

## 5. Select on dev

Expected dev WER (base model with the same prompt `p4`: 13.69%):

| step | 40 | 120 |
|---|---:|---:|
| dev WER | 12.84% | **12.70%** |

Most of the gain is there at the first checkpoint: EdAcc's conventions, learned within 40 steps. `val_loss` rises
from step 40 on (1.45 → 1.87 at step 200) while dev WER still edges down. Step 120 is selected.

In [ ]:
dev = {}
for step in (40, 120):
    out = EXPORT / f"acc_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'acc'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_acc_{step}.log")
    dev[step] = 100 * evaluate(out, d / "dev_ft.json", f"acc_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev WER {dev[step]:.2f}%")
BEST = min(dev, key=dev.get)

## 6. Test, once

In [ ]:
MODEL = EXPORT / f"acc_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "acc_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "acc_librispeech", "--task asr --normalizer english")
FT_WER = 100 * ft["score"]
print(f"test WER: base {BASELINE:.2f}% → fine-tuned {FT_WER:.2f}% ({(BASELINE - FT_WER) / BASELINE:+.1%} relative reduction); "
      f"LibriSpeech {100 * guard['score']:.2f}%")
by_length("acc_TEST")
result("13 baseline test WER", BASELINE, 18.37, 0.5)
result("13 fine-tuned test WER", FT_WER, 14.96, 0.6)

Expected results:

| test WER | < 3 s | 3-10 s | 10-30 s | > 30 s | **all** | LibriSpeech WER |
|---|---:|---:|---:|---:|---:|---:|
| base, prompt chosen on dev (p1) | 49.6% | 16.9% | 11.2% | 16.7% | 18.37% | 1.54% |
| base, test-oracle prompt (p4) | 48.8% | 16.9% | 10.6% | 10.0% | 16.20% | 1.54% |
| **fine-tuned (step 120)** | 41.5% | 15.0% | 11.4% | 9.1% | **14.96%** | 1.49% |

**−18.6% relative** to the baseline (prompt chosen on dev); −7.7% against the test-oracle prompt. On dev, the same
model improves WER by 6.0% (13.51% → 12.70%). The fine-tune helps where the conventions live (short clips,
48.8% → 41.5% against `p4`) and gives a little back on 10-30 s segments (more insertions), so most of the total gain
comes from the short clips.

Three other levers do not improve dev: adding short backchannel clips to training, adding ~20 h of AMI meeting data,
and training at LR 1e-4. System combination, e.g. ROVER voting over several fine-tuned systems
([`rover_combine.py`](../../../examples/speechlm2/finetuning/rover_combine.py)), is the remaining lever to try.

One side effect to know about: this fine-tune is trained on a single prompt. With **no** prompt at all, it may answer
question-like LibriSpeech sentences ("why are we to be divided") with essays instead of transcribing them. With the
transcription prompt used above, LibriSpeech WER is unchanged. If your users may send no instruction, train with
several prompt phrasings.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="acc")

## Takeaways

1. **Grep the references for scoring markers.** Left in, `IGNORE_TIME_SEGMENT_IN_SCORING` turns a marginal improvement
   into an apparent large one. When a gain concentrates in a few rows, read the rows with the largest errors.
2. **Cap the output length by audio duration** on any ASR task with a large `max_tokens`, for the baseline too.
3. **Name the language in the prompt**, even for English.
4. **Some benchmarks are about conventions, not acoustics.** Once the base model is good, a fine-tune mostly learns the
   annotators' habits, and backchannels in 0.3-second clips have a floor that more steps don't move.